# Toy Inference From Run Directory

Interactive notebook version for continuing development.

Workflow:
1. Set `RUN_DIR` and toy physical inputs.
2. Load `train_config_used.json`, `scaler_state.json`, and checkpoint.
3. Build scaled toy features.
4. Run inference and plot mean plus `+/- 1 std`.

In [ ]:
from __future__ import annotations

import json
from pathlib import Path
from typing import Any

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn

In [ ]:
def make_activation(name: str) -> nn.Module:
    name = (name or "relu").lower()
    if name == "relu":
        return nn.ReLU()
    if name == "gelu":
        return nn.GELU()
    if name == "tanh":
        return nn.Tanh()
    raise ValueError(f"Unsupported activation '{name}'")


class PenetrationMLP(nn.Module):
    def __init__(
        self,
        input_dim: int,
        hidden_dims: list[int],
        output_dim: int,
        *,
        activation: str = "relu",
        dropout: float = 0.0,
    ) -> None:
        super().__init__()
        layers: list[nn.Module] = []
        in_dim = input_dim
        for hidden_dim in hidden_dims:
            layers.extend(
                [
                    nn.Linear(in_dim, hidden_dim),
                    nn.LayerNorm(hidden_dim),
                    make_activation(activation),
                ]
            )
            if dropout > 0:
                layers.append(nn.Dropout(dropout))
            in_dim = hidden_dim
        layers.append(nn.Linear(in_dim, output_dim))
        self.net = nn.Sequential(*layers)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


def split_mu_logvar(model_output: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
    mu, log_var = model_output.chunk(2, dim=-1)
    return mu, log_var


def resolve_model_path(run_dir: Path) -> Path:
    for name in ("best_model_stage2.pt", "best_model_stage1.pt"):
        model_path = run_dir / name
        if model_path.exists():
            return model_path
    raise FileNotFoundError(f"No supported model checkpoint found under: {run_dir}")


def zscore_from_state(value: float, z_col: str, scaler_state: dict[str, Any]) -> float:
    stats = scaler_state["zscore"][z_col]
    return (float(value) - float(stats["mean"])) / (float(stats["std"]) + 1e-12)


def build_toy_feature_matrix(
    raw: dict[str, float],
    time_ms: np.ndarray,
    scaler_state: dict[str, Any],
    feature_columns: list[str],
    time_feature: str,
) -> np.ndarray:
    p_inj = float(raw["injection_pressure_bar"])
    p_ch = float(raw["chamber_pressure_bar"])
    delta_p = max(p_inj - p_ch, 1e-6)

    time_min_ms = float(scaler_state["time"]["min_ms"])
    time_max_ms = float(scaler_state["time"]["max_ms"])
    time_span_ms = max(time_max_ms - time_min_ms, 1e-12)
    time_norm = np.clip((time_ms - time_min_ms) / time_span_ms, 0.0, 1.0).astype(np.float32)

    feature_series: dict[str, np.ndarray] = {
        time_feature: time_norm,
        "tilt_angle_radian_z": np.full_like(
            time_norm,
            zscore_from_state(raw["tilt_angle_radian"], "tilt_angle_radian_z", scaler_state),
            dtype=np.float32,
        ),
        "plumes_z": np.full_like(
            time_norm,
            zscore_from_state(raw["plumes"], "plumes_z", scaler_state),
            dtype=np.float32,
        ),
        "diameter_mm_z": np.full_like(
            time_norm,
            zscore_from_state(raw["diameter_mm"], "diameter_mm_z", scaler_state),
            dtype=np.float32,
        ),
        "injection_duration_us_z": np.full_like(
            time_norm,
            zscore_from_state(raw["injection_duration_us"], "injection_duration_us_z", scaler_state),
            dtype=np.float32,
        ),
        "log_injection_pressure_bar_z": np.full_like(
            time_norm,
            zscore_from_state(np.log(p_inj), "log_injection_pressure_bar_z", scaler_state),
            dtype=np.float32,
        ),
        "log_chamber_pressure_bar_z": np.full_like(
            time_norm,
            zscore_from_state(np.log(max(p_ch, 1e-6)), "log_chamber_pressure_bar_z", scaler_state),
            dtype=np.float32,
        ),
        "log_delta_pressure_bar_z": np.full_like(
            time_norm,
            zscore_from_state(np.log(delta_p), "log_delta_pressure_bar_z", scaler_state),
            dtype=np.float32,
        ),
        "control_backpressure_bar_z": np.full_like(
            time_norm,
            zscore_from_state(raw["control_backpressure_bar"], "control_backpressure_bar_z", scaler_state),
            dtype=np.float32,
        ),
    }

    columns: list[np.ndarray] = []
    for name in feature_columns:
        if name not in feature_series:
            raise KeyError(f"Unsupported feature column in config: {name}")
        columns.append(feature_series[name])
    return np.column_stack(columns).astype(np.float32)

In [ ]:
# User config
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "MLP").exists() and PROJECT_ROOT.name == "MLP":
    PROJECT_ROOT = PROJECT_ROOT.parent

RUN_DIR = PROJECT_ROOT / "MLP" / "runs_mlp" / "stage2_NLL_penetration_20260317_194155"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

toy_raw = {
    "tilt_angle_radian": float(np.deg2rad(20.0)),
    "plumes": 10.0,
    "diameter_mm": 0.355,
    "injection_duration_us": 800.0,
    "injection_pressure_bar": 2000.0,
    "chamber_pressure_bar": 5.0,
    "control_backpressure_bar": 4.0,
}

toy_n_points = 300
toy_time_ms = np.linspace(0.0, 5.0, toy_n_points, dtype=np.float32)

In [ ]:
config_path = RUN_DIR / "train_config_used.json"
scaler_path = RUN_DIR / "scaler_state.json"
model_path = resolve_model_path(RUN_DIR)

with config_path.open("r", encoding="utf-8") as f:
    train_config = json.load(f)
with scaler_path.open("r", encoding="utf-8") as f:
    scaler_state = json.load(f)

feature_columns = list(train_config["feature_columns"])
time_feature = str(train_config.get("time_feature", "time_norm_0_5ms"))

model = PenetrationMLP(
    input_dim=int(train_config["input_dim"]),
    hidden_dims=[int(x) for x in train_config["hidden_dims"]],
    output_dim=int(train_config["output_dim"]),
    activation=str(train_config.get("activation", "relu")),
    dropout=float(train_config.get("dropout", 0.0)),
)
state = torch.load(model_path, map_location=DEVICE)
model.load_state_dict(state)
model.to(DEVICE)
model.eval()

print(f"RUN_DIR: {RUN_DIR.resolve()}")
print(f"Model checkpoint: {model_path.name}")
print(f"Device: {DEVICE}")
print(f"Feature dimension: {len(feature_columns)}")

In [ ]:
toy_features_np = build_toy_feature_matrix(
    raw=toy_raw,
    time_ms=toy_time_ms,
    scaler_state=scaler_state,
    feature_columns=feature_columns,
    time_feature=time_feature,
)
toy_features = torch.as_tensor(toy_features_np, dtype=torch.float32, device=DEVICE)

with torch.no_grad():
    toy_out = model(toy_features)
    toy_mu, toy_log_var = split_mu_logvar(toy_out)

toy_mu_np = toy_mu.detach().cpu().numpy().reshape(-1)
toy_log_var_np = toy_log_var.detach().cpu().numpy().reshape(-1)
std_floor = float(train_config.get("std_clamp_min", 0.0))
toy_std_np = np.maximum(np.sqrt(np.exp(toy_log_var_np)), std_floor)
toy_upper_np = toy_mu_np + toy_std_np
toy_lower_np = toy_mu_np - toy_std_np

print("Toy inference completed with feature shape:", toy_features_np.shape)
print("Expected feature dimension:", len(feature_columns))
print("Predicted std range:", float(np.min(toy_std_np)), float(np.max(toy_std_np)))

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(toy_time_ms, toy_mu_np, linewidth=2, label="Predicted mean")
plt.fill_between(toy_time_ms, toy_lower_np, toy_upper_np, alpha=0.25, label="Predicted +/- 1 std")
plt.plot(toy_time_ms, 90 * np.ones_like(toy_time_ms), linestyle="--", label="90 mm reference")
plt.xlabel("Time (ms)")
plt.ylabel("Predicted penetration")
plt.title("Toy Inference: 0-5 ms Penetration Sweep with Uncertainty")
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.ylim(0, 250)
plt.xlim(0, 5)
plt.show()